# 03 — Topic modelling (LDA)

Cluster the residual (non-government, non-candidate) ad bodies into latent topics so we can:

- Filter out commercial/spam topics (fashion, fitness apps, retail) from the political-adjacent subset.
- Surface what *kinds* of political messaging are circulating outside party/candidate channels — climate, cost-of-living, Voice, housing, etc.
- Layer topic labels into the v3 parquet for cross-tabbing with sentiment (notebook 04) and spend/impressions.

Approach: fit a single LDA model on the full residual corpus, eyeball the top words per topic, hand-label each topic in a CSV, join the labels back to the corpus.

## 1. Preprocessing

Load v2 parquet, filter to one row per ad (`ad_seq_no = 1`) and non-classified (`match_type IS NULL`). Extract the first creative body, tokenise with `RegexTokenizer`, drop stop words (English defaults + domain-specific noise). Vectorise word counts with `CountVectorizer` — raw counts, not TF-IDF, since LDA expects integer term frequencies.

Cache the vectorised features so subsequent LDA fits (at different `k`) don't re-run preprocessing.

### 1.1 Spark session

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    array_contains, col, coalesce, concat_ws, expr, length, lit,
)

# Same parquet-committer override as notebook 02 — cluster default points at an EMR
# class whose JAR isn't on the classpath.
spark = SparkSession.builder \
    .appName('FB_API_topics') \
    .config('spark.sql.parquet.output.committer.class',
            'org.apache.parquet.hadoop.ParquetOutputCommitter') \
    .config('mapreduce.fileoutputcommitter.algorithm.version', '2') \
    .getOrCreate()

print('Master:', spark.sparkContext.master)
print('Spark version:', spark.version)

Master: yarn
Spark version: 3.5.0


26/05/11 05:14:53 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


### 1.2 Paths

In [2]:
V2_PATH           = '/user/s3348393/main/preprocessing/v2/parquet'
INTERMEDIATE_PATH = '/user/s3348393/main/preprocessing/v3/intermediate_parquet'  # corpus + topic_id
V3_PATH           = '/user/s3348393/main/preprocessing/v3/parquet'               # final with labels

TOPIC_TERMS_CSV   = '../data/topic_terms.csv'    # output: top words per topic (for human review)
TOPIC_LABELS_CSV  = '../data/topic_labels.csv'   # input: human-edited labels

### 1.3 Load v2 and filter to the residual corpus

Keep one row per ad (`ad_seq_no = 1`) and only ads that:

- didn't classify as candidate/party/government (`match_type IS NULL`)
- are English-language (`array_contains('languages', 'en')`) — drops non-English clusters
- aren't from a byline we've explicitly tagged as commercial/non-political (`COMMERCIAL_BYLINES`) — iteratively grown as LDA surfaces noise topics

Body-text extraction is more inclusive than just `creative_bodies[0]`:

1. `first_non_empty(arr)` returns the **first non-null, non-empty** element of an array column — recovers ads where the first creative-body entry is null but a later one is real (multi-variant ads), or where the singular `ad_creative_body` was null but other text fields are populated.
2. Concatenate `creative_bodies` + `creative_link_descs` + `creative_link_titles` into a single document — more text = better LDA signal, and ads with no body but a populated link description ("Sign the petition", "Donate now") still contribute.
3. Drop ads with no text in any of these fields — LDA can't process empty documents.

`creative_link_captions` is skipped because it's usually just a domain name (low signal, noise).

In [9]:
df = spark.read.parquet(V2_PATH)
print('All v2 rows:    ', df.count())


def first_non_empty(col_name):
    """First non-null, non-empty element of an array column. Returns null if none."""
    return expr(f"filter({col_name}, x -> x is not null and length(x) > 0)[0]")


# Bylines we've identified as clearly non-political (commercial, recruitment, etc.)
# and want excluded from the LDA corpus. Iteratively grown — add bylines here when
# LDA surfaces them as dominating a noise/commercial topic.
COMMERCIAL_BYLINES = {
    'Access',                            # Indigenous Employment Australia — job listings
    'Streamotion Pty Ltd',               # Kayo / Binge sports + entertainment streaming
    'SBS Australia',                     # SBS On Demand streaming promotions
    'SBS Arabic24',                      # SBS language-stream marketing
    'SBS Mandarin中文普通话',            # SBS language-stream marketing
    'The Squiz',                         # paid news newsletter
    'Hair Cooki'                           #hair care ad
}


corpus = df.filter(
        (col('ad_seq_no') == 1) &
        col('match_type').isNull() &
        # Language: keep ads where languages is null (untagged — ~96% of residual)
        # OR explicitly tagged as English. Drops the small confirmed-non-English tail.
        (col('languages').isNull() | array_contains('languages', 'en')) &
        ~col('bylines').isin(list(COMMERCIAL_BYLINES))
    ) \
    .withColumn('body_text',  first_non_empty('creative_bodies')) \
    .withColumn('desc_text',  first_non_empty('creative_link_descs')) \
    .withColumn('title_text', first_non_empty('creative_link_titles')) \
    .withColumn('body',
        concat_ws(' ',
            coalesce(col('body_text'),  lit('')),
            coalesce(col('desc_text'),  lit('')),
            coalesce(col('title_text'), lit('')),
        )
    ) \
    .filter(length(col('body')) > 0) \
    .drop('body_text', 'desc_text', 'title_text')

print('Residual corpus:', corpus.count())
corpus.select('page_name', 'bylines', 'body').show(3, truncate=80)

All v2 rows:     3128023


Residual corpus: 49467
+--------------------+--------------------+--------------------------------------------------------------------------------+
|           page_name|             bylines|                                                                            body|
+--------------------+--------------------+--------------------------------------------------------------------------------+
|      Thrive by Five|      Thrive By Five|Every family wants to make sure their children have the best start in life. \...|
|          Talk Black|              GetUp!|The Federal Election is around the corner. To have your say on the issues you...|
|Australian Democrats|Australian Democrats|Vote for the Australian Democrats in the Senate to support immediate climate ...|
+--------------------+--------------------+--------------------------------------------------------------------------------+
only showing top 3 rows



### 1.4 Stop words

English defaults plus a small list of domain-specific noise: URL fragments, generic call-to-action words. Keep this list deliberately short — `minDF` and `maxDF` in `CountVectorizer` will handle most of the frequency-based filtering automatically. Iterate after the first LDA fit if specific tokens are dominating topics with no signal.

In [10]:
from pyspark.ml.feature import StopWordsRemover

stop_words = StopWordsRemover.loadDefaultStopWords('english') + [
    # URL / web junk that survives tokenisation
    'https', 'http', 'www', 'com', 'org', 'au', 'co', 'html',
    # contraction fragments surviving minTokenLength=2
    're', 've', 'll',
    # generic fillers (high frequency, low topic-discrimination value)
    'help', 'time', 'like', 'need', 'make', 'take', 'people',
    'year', 'years', 'today', 'also', 'will', 'can', 'get',
    'see', 'know', 'one', 'two', 'new', 'now', 'us',
    # generic CTA (kept short — don't strip 'petition', 'donate', 'sign', etc.
    # since those carry topic signal)
    'click', 'learn',
]

print('Stop-words list size:', len(stop_words))

Stop-words list size: 215


### 1.5 Preprocessing pipeline

Three stages: `RegexTokenizer` (split on `\W+`, lowercase, drop tokens shorter than 2 chars) → `StopWordsRemover` (English defaults + domain noise) → `CountVectorizer` (vocab≤5,000, term must appear in ≥100 ads, term must appear in ≤30% of ads).

`minDF=100` is tighter than the conventional default — it compresses the long-tail vocabulary used by individual small commercial advertisers, forcing LDA to cluster around vocabulary that's actually shared across the political-advocacy ecosystem. Niche-but-real political terms (`woodside`, `quoll`, `uyghur`) should still clear the threshold; one-off commercial product names won't.

Wrap in a `Pipeline` so we can `.fit().transform()` in one go and have a single fitted artefact to introspect afterwards.

In [11]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import RegexTokenizer, CountVectorizer

tokenizer = RegexTokenizer(
    inputCol='body', outputCol='raw_tokens',
    pattern=r'\W+', toLowercase=True, minTokenLength=2,
)

remover = StopWordsRemover(
    inputCol='raw_tokens', outputCol='tokens',
    stopWords=stop_words,
)

vectorizer = CountVectorizer(
    inputCol='tokens', outputCol='features',
    vocabSize=5000,
    minDF=100,    # term must appear in ≥100 ads — bumped from 50 to compress long-tail vocab
    maxDF=0.3,    # term appearing in >30% of ads is dropped (auto stop-word filter)
)

prep_pipeline = Pipeline(stages=[tokenizer, remover, vectorizer])

### 1.6 Fit, transform, cache

Fit the pipeline once. The output `features_df` carries every original column plus `raw_tokens`, `tokens`, and `features` (the sparse count vector LDA consumes). Cache it so the k-sweep in section 2 doesn't re-run preprocessing per `k`.

The `.count()` call forces Spark to actually materialise the cache — without it, the cache is registered lazily and nothing happens until something else triggers an action.

In [12]:
prep_model  = prep_pipeline.fit(corpus)
features_df = prep_model.transform(corpus).cache()

print('Cached rows:    ', features_df.count())

vocab = prep_model.stages[-1].vocabulary
print('Vocabulary size:', len(vocab))
print('\nTop 30 vocabulary terms (most frequent first):')
print(vocab[:30])

Cached rows:     49467
Vocabulary size: 2578

Top 30 vocabulary terms (most frequent first):
['sign', 'australia', 'climate', 'government', 'support', 'australian', 'petition', 'vote', 'election', 'change', 'women', 'community', 'protect', 'woodside', 'future', 'action', 'local', 'world', 'federal', 'join', 'stop', 'free', 'energy', 'life', 'children', 'donate', 'early', 'power', 'australians', 'demand']


## 2. Explore `k` on a sample

Fit LDA at several `k` values (5, 10, 15, 20) on a 10% sample of the cached features. Print top-12 words per topic for each `k`. Eyeball the printouts to pick a `k` where topics are distinct and each list reads as a coherent theme.

Fix `seed=42` so comparing `k=10` vs `k=15` isn't muddled by random init differences. Cheap and disposable — no parquet writes from this section.

In [ ]:
from pyspark.ml.clustering import LDA

# 10% sample of the cached features. Cache the sample too — the four LDA fits
# below will scan it repeatedly. seed=42 keeps the sample composition stable.
sample_df = features_df.sample(0.1, seed=42).cache()
print(f'Sample size: {sample_df.count():,}')

# Lookup from CountVectorizer integer term indices back to readable words.
vocab = prep_model.stages[-1].vocabulary

# Fit LDA at each k. seed=42 fixed so k=5 vs k=10 etc. are comparable.
for k in [5, 10, 15, 20]:
    print(f'\n=== k = {k} ===')
    lda = LDA(featuresCol='features', k=k, maxIter=20, seed=42)
    model = lda.fit(sample_df)
    topics = model.describeTopics(maxTermsPerTopic=12).collect()
    for row in topics:
        words = ' '.join(vocab[i] for i in row.termIndices)
        print(f'  Topic {row.topic:>2}: {words}')

sample_df.unpersist()

## 3. Final fit on full corpus

Refit LDA at `k=20` on the full cached features (no sampling). Transform the corpus to attach `topicDistribution` (length-20 vector) and `topic_id` (argmax) to every ad. Then persist two artefacts:

- **Intermediate parquet** — full corpus + LDA columns. Expensive to recompute (LDA on 50k docs at k=20 takes ~30–60s); write once so the labelling round-trip doesn't refit.
- **`data/topic_terms.csv`** — one row per topic with `top_terms` (top 15 words) and `top_bylines` (top 5 advertisers by ad count). The bylines column is the key labelling aid: combined with the term list, you can usually tell whether a topic is e.g. *Climate Council reef campaign* (top byline: Climate Council) vs *Greenpeace anti-Woodside* (top byline: Greenpeace) vs *commercial residue* (top byline: a brand) — much faster than guessing from terms alone.

The CSV also has an empty `label` column for you to fill in. Save as `data/topic_labels.csv` when done.

In [15]:
from pyspark.ml.clustering import LDA
from pyspark.ml.functions import vector_to_array
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, desc

K = 20

# Fit LDA at k=K on the full cached features. seed=42 for reproducibility.
lda = LDA(featuresCol='features', k=K, maxIter=20, seed=42)
lda_model = lda.fit(features_df)

# Transform: attach topicDistribution and dominant topic_id to every ad.
classified = lda_model.transform(features_df) \
    .withColumn('topic_array', vector_to_array('topicDistribution')) \
    .withColumn('topic_id', expr('array_position(topic_array, array_max(topic_array)) - 1'))

# Vocabulary for term-index -> word translation.
vocab = prep_model.stages[-1].vocabulary

# Top 15 terms per topic from describeTopics.
topics_rows = lda_model.describeTopics(maxTermsPerTopic=15).collect()
topic_terms = {row.topic: [vocab[i] for i in row.termIndices] for row in topics_rows}

# Top 5 bylines per topic — Spark window function over (topic_id, bylines, count).
w = Window.partitionBy('topic_id').orderBy(desc('count'))
top_bylines_rows = classified.filter(col('bylines').isNotNull()) \
    .groupBy('topic_id', 'bylines').count() \
    .withColumn('rank', row_number().over(w)) \
    .filter(col('rank') <= 5) \
    .orderBy('topic_id', 'rank') \
    .collect()

top_bylines = {}
for row in top_bylines_rows:
    top_bylines.setdefault(row.topic_id, []).append(row.bylines)

# Print to console — easy to scan while you draft labels.
print(f'k = {K}\n')
for tid in range(K):
    words = ' '.join(topic_terms.get(tid, []))
    bls   = ' | '.join(top_bylines.get(tid, []))
    print(f'Topic {tid:>2}:')
    print(f'  Terms:   {words}')
    print(f'  Bylines: {bls}')
    print()

26/05/11 05:26:45 WARN OnlineLDAOptimizer: The input data is not directly cached, which may hurt performance if its parent RDDs are also uncached.


k = 20

Topic  0:
  Terms:   better election candidates heard demand transport local tell deserve animal getting alliance welfare hunter fair
  Bylines: Australian Automobile Association | Animals Australia | Amnesty International Australia | Hunter Jobs Alliance | Australian Unions

Topic  1:
  Terms:   community government local australia state council mp parliament australian nsw road member south news 2022
  Bylines: The Pharmacy Guild of Australia | South Australian Liberals | OZ Arab Media | Private Media | Armenian National Committee of Australia

Topic  2:
  Terms:   whales gas woodside donate project electricity campaign toxic marine australia climate stop life greenpeace donation
  Bylines: Greenpeace Australia Pacific | The Wilderness Society | Nature Conservation Council | Advance Australia | Australian Conservation Foundation

Topic  3:
  Terms:   care health aged ukraine support government medical workers families needs morrison australian crisis product refugees
  Byline

In [16]:
import pandas as pd

# Write topic_terms.csv — one row per topic with top terms + top bylines + empty
# label column for the human to fill in.
topic_terms_pdf = pd.DataFrame([
    {
        'topic_id':    tid,
        'top_terms':   ' '.join(topic_terms.get(tid, [])),
        'top_bylines': '|'.join(top_bylines.get(tid, [])),
        'label':       '',
    }
    for tid in range(K)
])
topic_terms_pdf.to_csv(TOPIC_TERMS_CSV, index=False)
print(f'Wrote {TOPIC_TERMS_CSV}')

# Write intermediate parquet — corpus + topic_id + topicDistribution.
# Drop the large intermediate columns (token arrays, sparse vectors) so the
# parquet is just the v2 corpus plus the LDA-derived columns.
intermediate = classified.drop('raw_tokens', 'tokens', 'features', 'topic_array')
spark.conf.set('spark.sql.parquet.output.committer.class',
               'org.apache.parquet.hadoop.ParquetOutputCommitter')
intermediate.write \
    .option('mapreduce.fileoutputcommitter.algorithm.version', '2') \
    .parquet(INTERMEDIATE_PATH, mode='overwrite')
print(f'Wrote {INTERMEDIATE_PATH}')

Wrote ../data/topic_terms.csv


Wrote /user/s3348393/main/preprocessing/v3/intermediate_parquet


## 4. Manual labelling (out-of-notebook)

Open `data/topic_terms.csv` in a spreadsheet, add a `label` column with human-readable names (e.g. `climate`, `cost_of_living`, `voice_referendum`, `commercial_retail`, `noise`). Save as `data/topic_labels.csv`.

Topics that look like commercial noise (fashion brands, fitness app keywords, etc.) get labels like `commercial_*` or `noise` — these are the categories notebook 04 / later filters will drop.

## 5. Join labels back

Read intermediate parquet + `data/topic_labels.csv`. Broadcast-join on `topic_id` to add a `topic_label` column. Write the result as v3 parquet — same schema as v2 with `topicDistribution`, `topic_id`, and `topic_label` appended.

Fully re-runnable: tweak labels, re-run this section, no LDA refit needed.